# HW2 retained best: 61.67% (592/960)

This notebook uses the exact 92,456-step checkpoint from the completed original-data full pass. It is the highest observed public score, rank 9/16 on 2026-10-17. It is not first place.

The notebook is prepared and statically checked but has not yet been run end to end. T4 timing and rerun output equality remain unverified. CPU preparation comes before GPU use. The generation cell saves results to Drive, checks exact output equality against the scored programs, records timing, and releases the GPU.

Pinned settings: complete input questions, output cap 1,280, batch 64, FP32 and strict greedy decoding; no repairs or alternate outputs. Different hardware or software can change generated outputs, which require a new verified bot submission.


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU for file preparation'
from google.colab import drive
drive.mount('/content/drive', timeout_ms=180000)

In [ ]:
import os, sys, json, hashlib, shutil, tarfile, subprocess
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Keep GPU off during file checks'
os.environ['USE_TF']='0'; os.environ['USE_FLAX']='0'
root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
assert Path('/content/drive').is_mount()
prepared = root/'prepared/inference-full-pass-092456-greedy-v1'
prepared.mkdir(parents=True, exist_ok=True)
work = Path('/content/hw2-inference-prepared')
work.mkdir(exist_ok=True)
def digest(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(4*1024*1024), b''): h.update(block)
    return h.hexdigest()
source = root/'prepared/selected-full-pass-v2'
source_ready = json.loads((source/'ready.json').read_text())
assert source_ready['source_commit'] == '82af82114fbf056a0487cc4fa0f759829c557bd9'
assert source_ready['source_sha256'] == '041a7a20491c15df8c9d647bac118de7be54d198f183495a516b6f1d3275d9d2'
assert digest(source/'source.tar') == source_ready['source_sha256']
with tarfile.open(source/'source.tar') as z: z.extractall(work, filter='data')
os.chdir(work)
sys.path.insert(0, str(work))
subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow==23.0.1','tokenizers==0.22.2','numpy==2.1.3','requests>=2.31'],check=True)
from homeworks.hw2.checkpoint_io import drive_preflight
from homeworks.hw2.data import fetch_parquet
drive_preflight(root)
parent = root/'runs/selected-full-pass-v2/step-092456'
manifest = json.loads((parent/'checkpoint.complete.json').read_text())
assert manifest['status'] == 'checkpoint_verified'
assert manifest['files']['model.safetensors'] == '0473efab1f978bc941ffe71474cc5541c52ba628d60077990d81eb635a2efa1b'
model_dir = work/'model'; model_dir.mkdir(exist_ok=True)
for name, expected in manifest['files'].items():
    assert Path(name).name == name
    if name != 'training_state.pt':
        assert digest(parent/name) == expected, name
        shutil.copyfile(parent/name, model_dir/name)
        assert digest(model_dir/name) == expected
shutil.copyfile(parent/'checkpoint.complete.json', model_dir/'checkpoint.complete.json')
scored = json.loads((parent/'results.complete.json').read_text())
assert scored['jsonl_sha256'] == '200f2018387d5e9fc48b174ec862df7f58466cf71dc3d47ae53a365e40f4de9d'
assert digest(parent/'predictions.jsonl') == scored['jsonl_sha256']
shutil.copyfile(parent/'predictions.jsonl', model_dir/'scored-predictions.jsonl')
test = fetch_parquet('data/test/test.parquet', work/'cache')
import pyarrow.parquet as pq
questions = pq.read_table(test,columns=['question'])['question'].to_pylist()
question_hash = hashlib.sha256(json.dumps(questions,ensure_ascii=False).encode()).hexdigest()
assert len(questions) == 1919
assert question_hash == '214746f8d4bc918b6fa1fde8ca3185a518a60637f239a2c45efa9a8143f612d3'
from transformers import PreTrainedTokenizerFast, T5ForConditionalGeneration
tokenizer = PreTrainedTokenizerFast.from_pretrained(model_dir, local_files_only=True)
model = T5ForConditionalGeneration.from_pretrained(model_dir, local_files_only=True)
assert sum(p.numel() for p in model.parameters()) == 42481152
del model
bundle = Path('/content/inference-inputs.tar')
with tarfile.open(bundle,'w') as z:
    for folder in ['homeworks','model','cache']:
        z.add(work/folder, arcname=folder)
    z.add(work/'source_commit.txt',arcname='source_commit.txt')
target = prepared/'inputs.tar'
shutil.copyfile(bundle,target)
assert digest(target) == digest(bundle)
ready = {'source_commit':source_ready['source_commit'],'bundle_sha256':digest(target),
         'model_sha256':manifest['files']['model.safetensors'],'questions_sha256':question_hash,
         'rows':1919,'cpu_load_verified':True}
(prepared/'ready.json').write_text(json.dumps(ready,indent=2))
assert json.loads((prepared/'ready.json').read_text()) == ready
print('CPU_INFERENCE_READY',ready)

## Generation
After CPU_INFERENCE_READY, use CPU or an available GPU. This cell verifies and restores inputs, generates strictly greedy programs, saves results to Drive, and releases its runtime. Do not run it in the active training notebook.


In [ ]:
from google.colab import runtime
try:
    from google.colab import drive
    drive.mount('/content/drive', timeout_ms=180000)
    from pathlib import Path
    import os, sys, json, hashlib, shutil, tarfile, subprocess, datetime
    from zipfile import ZipFile, ZIP_DEFLATED
    os.environ['USE_TF']='0'; os.environ['USE_FLAX']='0'
    root=Path('/content/drive/MyDrive/HSE_NLP_HW2')
    prepared=root/'prepared/inference-full-pass-092456-greedy-v1'
    ready=json.loads((prepared/'ready.json').read_text())
    assert ready['cpu_load_verified'] and ready['rows']==1919
    assert ready['source_commit']=='82af82114fbf056a0487cc4fa0f759829c557bd9'
    work=Path('/content/hw2-inference-greedy');work.mkdir(exist_ok=True)
    bundle=Path('/content/inference-inputs.tar')
    shutil.copyfile(prepared/'inputs.tar',bundle)
    assert hashlib.sha256(bundle.read_bytes()).hexdigest()==ready['bundle_sha256']
    with tarfile.open(bundle) as z:z.extractall(work,filter='data')
    os.chdir(work);sys.path.insert(0,str(work))
    subprocess.run([sys.executable,'-m','pip','-q','install','transformers==4.57.6','pyarrow==23.0.1','tokenizers==0.22.2','numpy==2.1.3','requests>=2.31'],check=True)
    from homeworks.hw2.checkpoint_io import sha256,drive_preflight
    from homeworks.hw2.inference import generate_predictions
    from homeworks.hw2.checkpoint_io import package_jsonl
    from homeworks.hw2.verify_export import verify
    drive_preflight(root)
    model_dir=work/'model'
    assert sha256(model_dir/'model.safetensors')==ready['model_sha256']
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    local=work/stamp/'step-092456'
    import torch
    torch.set_num_threads(min(4, torch.get_num_threads()))
    assert not local.exists(), 'Preserve existing exports'
    count, seconds = generate_predictions(model_dir,local/'predictions.csv',work/'cache',
        batch_size=64,max_input_tokens=0,max_new_tokens=1280,
        num_beams=1,syntax_retry=False,precision='fp32')
    digest = package_jsonl(local/'predictions.csv',local/'predictions.jsonl')
    for filename in ['checkpoint.complete.json','step.json','config.json','generation_config.json',
                     'tokenizer.json','tokenizer_config.json','special_tokens_map.json','run_config.json']:
        shutil.copyfile(model_dir/filename,local/filename)
    result = dict(step=json.loads((model_dir/'step.json').read_text())['step'],rows=count,
        jsonl_sha256=digest,generation_seconds=seconds,source_commit=ready['source_commit'],
        selection='greedy',bot_status='pending',t4_timing_verified=False)
    (local/'results.complete.json').write_text(json.dumps(result,indent=2))
    evidence=verify(local,model_dir,ready['source_commit'])
    assert evidence['questions_sha256']==ready['questions_sha256']
    provenance=json.loads((local/'predictions.provenance.json').read_text())
    assert provenance['num_beams']==1 and provenance['syntax_retry_beams']==0
    assert provenance['selection']=='greedy'
    assert provenance['max_input_tokens'] is None and provenance['max_new_tokens']==1280
    assert provenance['batch_size']==64 and provenance['precision']=='torch.float32'
    evidence['matches_scored_programs']=(local/'predictions.jsonl').read_bytes()==(model_dir/'scored-predictions.jsonl').read_bytes()
    evidence['runtime_versions']={name:__import__(name).__version__ for name in ['torch','transformers','tokenizers','numpy','pyarrow']}
    gpu=provenance['gpu']
    evidence['t4_timing_verified']=bool(gpu and 'T4' in gpu)
    evidence['t4_within_180_seconds']=(provenance['generation_seconds']<=180
                                      if evidence['t4_timing_verified'] else None)
    (local/'timing.json').write_text(json.dumps(evidence,indent=2))
    output=root/'inference'/('step-092456-greedy-'+stamp)
    output.mkdir(parents=True,exist_ok=False)
    for path in local.iterdir():
        shutil.copyfile(path,output/path.name)
        assert sha256(path)==sha256(output/path.name)
    # Build the exported archive only from verified Drive copies.
    with ZipFile(output/'inference-results.zip','w',ZIP_DEFLATED) as z:
        for path in sorted(output.iterdir()):
            if path.suffix in {'.json','.jsonl','.csv'}:z.write(path,arcname=path.name)
    (prepared/'last-result.json').write_text(json.dumps({'path':str(output)}))
    print('SAVED_TO_DRIVE',str(output),evidence)
finally:
    try:runtime.unassign()
    except Exception as error:
        print('Release failed: disconnect and delete runtime manually.',type(error).__name__)


## Export from Drive
Use the authenticated Drive CLI or the CPU cell below. Verify hashes and exact token provenance before submitting once to the bot. Preserve the receipt beside the exported checkpoint.


In [ ]:
from pathlib import Path
assert not Path('/dev/nvidia0').exists(), 'Select CPU before exporting'
from google.colab import drive, files
drive.mount('/content/drive', timeout_ms=180000)
import json
root = Path('/content/drive/MyDrive/HSE_NLP_HW2')
output = Path(json.loads((root/'prepared/inference-full-pass-092456-greedy-v1/last-result.json').read_text())['path'])
assert output.is_relative_to(root/'inference')
print(json.loads((output/'results.complete.json').read_text()))
files.download(str(output/'inference-results.zip'))